# 🔍 Deepfake Detection — CNN + Transformer
### Notebook d'entraînement — Version corrigée complète

In [ ]:
# ==========================================
# CELLULE 0 — DÉCOMPRESSION DU DATASET
# (Ne lancer qu'une seule fois)
# ==========================================
# !unzip DFDC_Faces_Dataset.zip -d data_extrait/

In [ ]:
# ==========================================
# CELLULE 1 — ÉQUILIBRAGE DES CLASSES
# (Ne lancer qu'une seule fois)
# ==========================================
import os
import random
import shutil

BASE_PATH = "/teamspace/studios/this_studio/data_extrait/DFDC_Faces_Dataset"

dossier_fake = os.path.join(BASE_PATH, "fake")
dossier_real = os.path.join(BASE_PATH, "real")

if not os.path.exists(dossier_fake):
    print(f"❌ Dossier fake introuvable : {dossier_fake}")
elif not os.path.exists(dossier_real):
    print(f"❌ Dossier real introuvable : {dossier_real}")
else:
    videos_fake = [f for f in os.listdir(dossier_fake) if os.path.isdir(os.path.join(dossier_fake, f))]
    videos_real = [f for f in os.listdir(dossier_real) if os.path.isdir(os.path.join(dossier_real, f))]

    print(f"📊 Fake avant : {len(videos_fake)}")
    print(f"📊 Real avant : {len(videos_real)}")

    target = len(videos_real)
    if len(videos_fake) > target:
        random.seed(42)
        nb_to_remove = len(videos_fake) - target
        print(f"🧹 Suppression de {nb_to_remove} dossiers fake...")
        paths_fake = [os.path.join(dossier_fake, v) for v in videos_fake]
        to_delete = random.sample(paths_fake, nb_to_remove)
        for path in to_delete:
            try:
                shutil.rmtree(path)
            except Exception as e:
                print(f"⚠️ Erreur suppression {path} : {e}")
        print("✅ Suppression terminée")
    else:
        print("✅ Pas besoin de suppression (déjà équilibré)")

    final_fake = len([f for f in os.listdir(dossier_fake) if os.path.isdir(os.path.join(dossier_fake, f))])
    final_real = len([f for f in os.listdir(dossier_real) if os.path.isdir(os.path.join(dossier_real, f))])
    print(f"📁 Fake après : {final_fake}")
    print(f"📁 Real après : {final_real}")

In [ ]:
# ==========================================
# CELLULE 2 — SÉPARATION TRAIN / TEST
# (Ne lancer qu'une seule fois)
# ==========================================
import os
import shutil
from sklearn.model_selection import train_test_split
from tqdm import tqdm

BASE_PATH = "/teamspace/studios/this_studio/data_extrait/DFDC_Faces_Dataset"

def split_data_in_place(base_path, test_size=0.2):
    real_dir = os.path.join(base_path, "real")
    fake_dir = os.path.join(base_path, "fake")

    if not os.path.exists(real_dir) or not os.path.exists(fake_dir):
        print(f"⚠️ Les dossiers 'real' et/ou 'fake' ne sont pas trouvés dans {base_path}.")
        print("Vérifie que la séparation n'a pas déjà été faite.")
        return

    for split in ['train', 'test']:
        for label in ['real', 'fake']:
            os.makedirs(os.path.join(base_path, split, label), exist_ok=True)

    for label in ['real', 'fake']:
        current_label_dir = os.path.join(base_path, label)
        video_folders = [f for f in os.listdir(current_label_dir)
                         if os.path.isdir(os.path.join(current_label_dir, f))]
        if len(video_folders) == 0:
            print(f"⚠️ Aucune vidéo trouvée dans {current_label_dir}")
            continue

        train_vids, test_vids = train_test_split(video_folders, test_size=test_size, random_state=42)

        for vid in tqdm(train_vids, desc=f"Déplacement vers Train - {label}"):
            src = os.path.join(current_label_dir, vid)
            dst = os.path.join(base_path, 'train', label, vid)
            shutil.move(src, dst)

        for vid in tqdm(test_vids, desc=f"Déplacement vers Test  - {label}"):
            src = os.path.join(current_label_dir, vid)
            dst = os.path.join(base_path, 'test', label, vid)
            shutil.move(src, dst)

        if len(os.listdir(current_label_dir)) == 0:
            os.rmdir(current_label_dir)

print("🚀 Début de la réorganisation des données...")
split_data_in_place(BASE_PATH, test_size=0.2)
print("\n✅ Séparation et déplacement terminés avec succès !")

In [ ]:
# ==========================================
# CELLULE 3 — IMPORTS
# ==========================================
import os
import math
import random
import numpy as np
from glob import glob
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score,
    accuracy_score, precision_score, recall_score, f1_score, average_precision_score
)
from tqdm import tqdm
import timm

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# ==========================================
# 1. CONFIGURATION
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device utilisé :", device)

torch.manual_seed(42)
if device == "cuda":
    torch.cuda.manual_seed_all(42)

BASE_PATH       = "/teamspace/studios/this_studio/data_extrait/DFDC_Faces_Dataset"
SAVE_DIR        = "/teamspace/studios/this_studio/pcd/models/"
os.makedirs(SAVE_DIR, exist_ok=True)

CHECKPOINT_PATH = os.path.join(SAVE_DIR, "checkpoint.pth")
BEST_MODEL_PATH = os.path.join(SAVE_DIR, "best_model.pth")
LAST_MODEL_PATH = os.path.join(SAVE_DIR, "last_model.pth")

seq_len = 4 # longueur de séquence (5 frames par séquence)

In [ ]:
# ==========================================
# 2. DATASETS
# ==========================================
class FaceDataset(Dataset):
    def __init__(self, base_path, transform=None):
        self.transform = transform
        self.data = []
        self.labels_dict = {}

        for label_name, label in [("real", 0), ("fake", 1)]:
            label_path = os.path.join(base_path, label_name)
            if not os.path.exists(label_path):
                continue
            for video_folder in sorted(os.listdir(label_path)):
                video_path = os.path.join(label_path, video_folder)
                if os.path.isdir(video_path):
                    for img_file in sorted(glob(os.path.join(video_path, "*.jpg"))):
                        self.data.append(img_file)
                        self.labels_dict[img_file] = label

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = self.data[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = self.labels_dict[img_path]
        return img, torch.tensor(label, dtype=torch.float32)


class FaceSequenceDataset(Dataset):
    def __init__(self, face_dataset, seq_len=5, selected_videos=None):
        self.dataset = face_dataset
        self.seq_len = seq_len
        self.sequences = []
        self.path_to_idx = {p: i for i, p in enumerate(self.dataset.data)}

        if selected_videos is not None:
            selected_videos = set(selected_videos)
        self.selected_videos = selected_videos

        video_dict = defaultdict(list)
        for img_path in self.dataset.data:
            class_name  = os.path.basename(os.path.dirname(os.path.dirname(img_path)))
            folder_name = os.path.basename(os.path.dirname(img_path))
            video_name  = f"{class_name}_{folder_name}"
            if (self.selected_videos is not None) and (video_name not in self.selected_videos):
                continue
            video_dict[video_name].append(img_path)

        for video_name, img_list in video_dict.items():
            img_list = sorted(img_list)
            if len(img_list) >= seq_len:
                # stride=2 pour réduire la redondance entre séquences
                for i in range(0, len(img_list) - seq_len + 1, 2):
                    self.sequences.append(img_list[i:i + seq_len])

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq_imgs_paths = self.sequences[idx]
        imgs = []
        labels = []
        try:
            for path in seq_imgs_paths:
                img_idx = self.path_to_idx[path]
                img_tensor, label = self.dataset[img_idx]
                imgs.append(img_tensor)
                labels.append(label)
            seq_tensor = torch.stack(imgs)
            label = labels[-1]
            return seq_tensor, label
        except Exception:
            new_idx = random.randint(0, len(self.sequences) - 1)
            return self.__getitem__(new_idx)

In [ ]:
# ==========================================
# 3. TRANSFORMS — AUGMENTATION RENFORCÉE
# ==========================================
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomRotation(10),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
# ==========================================
# 4. PRÉPARATION DES DONNÉES
# ==========================================
TRAIN_PATH = os.path.join(BASE_PATH, "train")
TEST_PATH  = os.path.join(BASE_PATH, "test")

train_face_dataset = FaceDataset(TRAIN_PATH, transform=train_transform)
test_face_dataset  = FaceDataset(TEST_PATH,  transform=test_transform)

def get_videos_from_dataset(dataset):
    video_names = set()
    for img_path in dataset.data:
        class_name  = os.path.basename(os.path.dirname(os.path.dirname(img_path)))
        folder_name = os.path.basename(os.path.dirname(img_path))
        video_names.add(f"{class_name}_{folder_name}")
    return list(video_names)

all_train_videos = get_videos_from_dataset(train_face_dataset)
test_videos      = get_videos_from_dataset(test_face_dataset)

# Séparation : 85% train / 15% val
train_videos, val_videos = train_test_split(all_train_videos, test_size=0.15, random_state=42)

train_seq_dataset = FaceSequenceDataset(train_face_dataset, seq_len=seq_len, selected_videos=train_videos)
val_seq_dataset   = FaceSequenceDataset(train_face_dataset, seq_len=seq_len, selected_videos=val_videos)
test_seq_dataset  = FaceSequenceDataset(test_face_dataset,  seq_len=seq_len, selected_videos=test_videos)

train_loader = DataLoader(
    train_seq_dataset, batch_size=32, shuffle=True,
    num_workers=4, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_seq_dataset, batch_size=32, shuffle=False,
    num_workers=4, pin_memory=True
)
test_loader = DataLoader(
    test_seq_dataset, batch_size=32, shuffle=False,
    num_workers=4, pin_memory=True
)

print(f"✅ Train sequences : {len(train_seq_dataset)}")
print(f"✅ Val sequences   : {len(val_seq_dataset)}")
print(f"✅ Test sequences  : {len(test_seq_dataset)}")

# Vérification équilibre
dossier_fake = os.path.join(BASE_PATH, "train", "fake")
dossier_real = os.path.join(BASE_PATH, "train", "real")
if os.path.exists(dossier_fake) and os.path.exists(dossier_real):
    print(f"Nombre de videos 'fake' : {len(os.listdir(dossier_fake))}")
    print(f"Nombre de videos 'real' : {len(os.listdir(dossier_real))}")

In [ ]:
# ==========================================
# 5. ARCHITECTURE DU MODÈLE
# ==========================================
class XceptionCNN(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.model = timm.create_model('xception', pretrained=pretrained)
        self.model.fc = nn.Identity()
        self.out_features = 2048

    def forward(self, x):
        return self.model(x)


class EfficientNetCNN(nn.Module):
    def __init__(self, model_name='efficientnet_b4', pretrained=True):
        super().__init__()
        self.model = timm.create_model(model_name, pretrained=pretrained)
        self.model.classifier = nn.Identity()
        self.out_features = 1792

    def forward(self, x):
        return self.model(x)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class CNN_Transformer_Deepfake(nn.Module):
    def __init__(
        self,
        xception_feat_dim=2048,
        efficient_feat_dim=1792,
        transformer_dim=512,
        num_heads=8,
        num_layers=3,
        num_classes=1,
        freeze_ratio=0.85,
        dropout_p=0.5
    ):
        super().__init__()
        self.xception  = XceptionCNN()
        self.efficient = EfficientNetCNN()

        self._freeze_partial(self.xception.model,  freeze_ratio)
        self._freeze_partial(self.efficient.model, freeze_ratio)

        combined_dim = xception_feat_dim + efficient_feat_dim
        self.feature_projection = nn.Sequential(
            nn.Linear(combined_dim, transformer_dim),
            nn.GELU(),
            nn.LayerNorm(transformer_dim),
            nn.Dropout(dropout_p)
        )

        self.pos_encoder = PositionalEncoding(transformer_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=transformer_dim,
            nhead=num_heads,
            dim_feedforward=transformer_dim * 4,
            dropout=dropout_p,
            activation="gelu",
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.classifier = nn.Sequential(
            nn.Linear(transformer_dim, 128),
            nn.GELU(),
            nn.Dropout(dropout_p),
            nn.Linear(128, num_classes)
        )

    def _freeze_partial(self, model, freeze_ratio):
        total_params = sum(1 for _ in model.parameters())
        freeze_until = int(total_params * freeze_ratio)
        for i, param in enumerate(model.parameters()):
            param.requires_grad = (i >= freeze_until)

    def forward(self, x_seq):
        batch, seq_len, C, H, W = x_seq.size()
        x_cnn = x_seq.view(batch * seq_len, C, H, W)

        feat_xcep = self.xception(x_cnn)
        feat_eff  = self.efficient(x_cnn)

        fused = torch.cat([feat_xcep, feat_eff], dim=1)
        fused = fused.view(batch, seq_len, -1)

        projected       = self.feature_projection(fused)
        transformer_in  = self.pos_encoder(projected)
        transformer_out = self.transformer(transformer_in)

        pooled = transformer_out.mean(dim=1)
        return self.classifier(pooled)

In [ ]:
# ==========================================
# 6. INITIALISATION DU MODÈLE ET OPTIMISEUR
# ==========================================

# ✅ Label smoothing manuel compatible BCE
class BCEWithLogitsLossSmoothed(nn.Module):
    def __init__(self, smoothing=0.05):
        super().__init__()
        self.smoothing = smoothing
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        # 1 → 0.975,  0 → 0.025
        targets_smooth = targets * (1 - self.smoothing) + self.smoothing / 2
        return self.bce(logits, targets_smooth)


model     = CNN_Transformer_Deepfake(freeze_ratio=0.85, dropout_p=0.5).to(device)
criterion = BCEWithLogitsLossSmoothed(smoothing=0.05)

# ✅ FIX CRITIQUE : list() pour éviter le générateur épuisé
trainable_params = list(filter(lambda p: p.requires_grad, model.parameters()))

optimizer = torch.optim.AdamW(trainable_params, lr=1e-4, weight_decay=1e-2)

# ✅ Warmup 2 epochs + CosineAnnealing : stable et sans plateau prématuré
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR

warmup_scheduler = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=2)
cosine_scheduler = CosineAnnealingLR(optimizer, T_max=18, eta_min=1e-6)
scheduler        = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[2]
)

num_epochs = 20
scaler     = torch.amp.GradScaler(device="cuda", enabled=(device == "cuda"))

print(f"✅ Modèle initialisé sur {device}")
print(f"   Paramètres entraînables : {sum(p.numel() for p in trainable_params):,}")

In [ ]:
# ==========================================
# 7. FONCTION D'ÉVALUATION
# ==========================================
def evaluate(model, data_loader, criterion, device):
    model.eval()
    total    = 0
    correct  = 0
    loss_sum = 0.0

    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device).unsqueeze(1)

            with torch.amp.autocast(device_type='cuda', enabled=(device == "cuda")):
                out  = model(batch_x)
                loss = criterion(out.float(), batch_y.float())

            if not torch.isnan(loss):
                loss_sum += loss.item()

            probs   = torch.sigmoid(out.float())
            preds   = (probs >= 0.5).float()
            correct += (preds == batch_y).sum().item()
            total   += batch_y.numel()

    avg_loss = loss_sum / len(data_loader) if len(data_loader) > 0 else 0
    acc      = correct / total if total > 0 else 0.0
    return avg_loss, acc

In [ ]:
# ==========================================
# 8. BOUCLE D'ENTRAÎNEMENT
# ==========================================
best_val_loss    = float('inf')
patience         = 7          # augmenté pour laisser le modèle explorer
patience_counter = 0
start_epoch      = 0

# Historique pour visualisation
history = {"train_loss": [], "val_loss": [], "val_acc": [], "lr": []}

if os.path.exists(CHECKPOINT_PATH):
    print(f"🔁 Checkpoint trouvé, chargement depuis : {CHECKPOINT_PATH}")
    checkpoint       = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    scheduler.load_state_dict(checkpoint["scheduler_state"])
    scaler.load_state_dict(checkpoint["scaler_state"])
    start_epoch      = checkpoint["epoch"] + 1
    best_val_loss    = checkpoint.get("best_val_loss", float('inf'))
    patience_counter = checkpoint.get("patience_counter", 0)
    history          = checkpoint.get("history", history)
    print(f"➡ Reprise à l'epoch {start_epoch + 1}/{num_epochs}")
else:
    print("🚀 Aucun checkpoint trouvé, démarrage d'un nouvel entraînement")

if len(train_loader) > 0:
    for epoch in range(start_epoch, num_epochs):
        model.train()
        epoch_loss = 0.0

        for batch_x, batch_y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device).unsqueeze(1)

            optimizer.zero_grad()

            with torch.autocast(device_type='cuda', enabled=(device == "cuda")):
                out  = model(batch_x)
                loss = criterion(out.float(), batch_y.float())

            scaler.scale(loss).backward()

            # Gradient clipping — empêche les explosions de gradient
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item()

        train_loss          = epoch_loss / len(train_loader)
        val_loss, val_acc   = evaluate(model, val_loader,  criterion, device)
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)

        # ✅ Scheduler step — SequentialLR (pas de métrique requise)
        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step()
        new_lr = optimizer.param_groups[0]['lr']
        if new_lr != old_lr:
            print(f"📉 LR : {old_lr:.2e} → {new_lr:.2e}")

        current_lr = optimizer.param_groups[0]['lr']

        # Historique
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["lr"].append(current_lr)

        print(
            f"Epoch {epoch+1}/{num_epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | "
            f"test_loss={test_loss:.4f} | test_acc={test_acc:.4f} | "
            f"lr={current_lr:.2e}"
        )

        # Early stopping basé sur val_loss
        if val_loss < best_val_loss - 1e-4:
            best_val_loss    = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            print(f"  ✅ Meilleur modèle sauvegardé (val_loss={best_val_loss:.4f})")
        else:
            patience_counter += 1
            print(f"  ⚠️  Pas d'amélioration (patience {patience_counter}/{patience})")
            if patience_counter >= patience:
                print("⏹ Early stopping déclenché.")
                break

        torch.save({
            "epoch":            epoch,
            "model_state":      model.state_dict(),
            "optimizer_state":  optimizer.state_dict(),
            "scheduler_state":  scheduler.state_dict(),
            "scaler_state":     scaler.state_dict(),
            "best_val_loss":    best_val_loss,
            "patience_counter": patience_counter,
            "num_epochs":       num_epochs,
            "history":          history,
        }, CHECKPOINT_PATH)
        print(f"💾 Checkpoint sauvegardé à l'epoch {epoch+1}")

    torch.save(model.state_dict(), LAST_MODEL_PATH)
    print("✅ Training terminé")

In [ ]:
# ==========================================
# 8b. VISUALISATION DES COURBES D'ENTRAÎNEMENT
# ==========================================
if history["train_loss"]:
    epochs_ran = range(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(epochs_ran, history["train_loss"], label="Train Loss")
    axes[0].plot(epochs_ran, history["val_loss"],   label="Val Loss")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(epochs_ran, history["val_acc"], color="green", label="Val Acc")
    axes[1].set_title("Validation Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    axes[1].grid(True)

    axes[2].plot(epochs_ran, history["lr"], color="orange", label="Learning Rate")
    axes[2].set_title("Learning Rate Schedule")
    axes[2].set_xlabel("Epoch")
    axes[2].set_yscale("log")
    axes[2].legend()
    axes[2].grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
# ==========================================
# 9. ÉVALUATION FINALE SUR LE TEST SET
# ==========================================
if os.path.exists(BEST_MODEL_PATH) and len(test_loader) > 0:
    print("\n⏳ Début de l'évaluation finale...")

    eval_model = CNN_Transformer_Deepfake(freeze_ratio=0.85, dropout_p=0.5).to(device)
    eval_model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
    eval_model.eval()

    test_loss_final = 0.0
    all_labels, all_probs, all_preds = [], [], []

    with torch.no_grad():
        for batch_x, batch_y in tqdm(test_loader, desc="Test final"):
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device).unsqueeze(1)

            with torch.amp.autocast(device_type='cuda', enabled=(device == "cuda")):
                logits = eval_model(batch_x)
                loss   = criterion(logits, batch_y)

            test_loss_final += loss.item()
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()

            all_labels.append(batch_y.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
            all_preds.append(preds.cpu().numpy())

    test_loss_final /= len(test_loader)
    all_labels = np.concatenate(all_labels).ravel()
    all_probs  = np.concatenate(all_probs).ravel()
    all_preds  = np.concatenate(all_preds).ravel()

    y_true  = all_labels.astype(int)
    y_pred  = all_preds.astype(int)
    y_score = all_probs

    print(f"\n📊 Test loss final : {test_loss_final:.4f}")
    print("\n📊 Résultats sur le test :")
    print(f" - Accuracy         : {accuracy_score(y_true, y_pred):.4f}")
    print(f" - Précision (fake) : {precision_score(y_true, y_pred, pos_label=1, zero_division=0):.4f}")
    print(f" - Rappel (fake)    : {recall_score(y_true, y_pred, pos_label=1, zero_division=0):.4f}")
    print(f" - F1-score (fake)  : {f1_score(y_true, y_pred, pos_label=1, zero_division=0):.4f}")

    if len(np.unique(y_true)) > 1:
        print(f" - AUC-ROC          : {roc_auc_score(y_true, y_score):.4f}")
        print(f" - AUC-PR           : {average_precision_score(y_true, y_score):.4f}")

    print("\nClassification report :")
    print(classification_report(
        y_true, y_pred,
        target_names=['real (0)', 'fake (1)'],
        zero_division=0
    ))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4, 4))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=['real (0)', 'fake (1)'],
        yticklabels=['real (0)', 'fake (1)']
    )
    plt.xlabel("Prédit")
    plt.ylabel("Vrai")
    plt.title("Matrice de confusion (test)")
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Aucun modèle trouvé ou test_loader vide.")